# 🧠 PyTorch Neuronales Netz — MNIST-Klassifikation

**Vollständiges Training mit `nn.Module`, DataLoader, Optimizer und GPU**

In diesem Notebook lernst du:
- `nn.Module` als Baustein für neuronale Netze
- `DataLoader` für effizientes Batch-Training
- `torch.optim` für Optimierungsalgorithmen
- Train/Test-Loop mit Evaluation
- Vergleich: MLP vs. CNN auf MNIST

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar:  {torch.cuda.is_available()}")

## 1. Device wählen

Automatisch GPU nutzen, wenn verfügbar — sonst CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 2. MNIST-Daten laden

MNIST: 70.000 handgeschriebene Ziffern (28×28 Pixel, Graustufen).
- 60.000 Trainingsbilder
- 10.000 Testbilder
- 10 Klassen (0–9)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                          # PIL → Tensor [0,1]
    transforms.Normalize((0.1307,), (0.3081,))     # Standardisierung (MNIST mean/std)
])

train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

print(f"Train: {len(train_data):,} | Test: {len(test_data):,}")

## 3. MLP (Multi-Layer Perceptron)

Ein einfaches vollvernetztes Netz: **784 → 128 → 64 → 10**

- `nn.Linear`: Vollständig verbundener Layer (y = xW^T + b)
- `F.relu`: ReLU-Aktivierung
- `nn.Dropout`: Regularisierung (schaltet zufällig Neuronen ab)
- `forward()`: Definiert den Datenfluss

In [ ]:
class SimpleMLP(nn.Module):
    """MLP: 784 → 128 → 64 → 10"""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # Flatten: (N, 28, 28) → (N, 784)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)           # Logits (kein Softmax — in CrossEntropyLoss)

mlp = SimpleMLP()
print(mlp)
print(f"\nParameter: {sum(p.numel() for p in mlp.parameters()):,}")

## 4. CNN (Convolutional Neural Network)

Faltungsnetz für Bilderkennung: **Conv → ReLU → MaxPool → Conv → ReLU → MaxPool → FC → FC**

- `nn.Conv2d`: 2D-Faltung (lernt räumliche Features)
- `nn.MaxPool2d`: Downsampling (reduziert Dimensionen)
- Nach 2× Pooling: 28→14→7, mit 64 Kanälen → 64×7×7 = 3136 Features

In [ ]:
class SimpleCNN(nn.Module):
    """CNN: 2×Conv + 2×FC"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # → (N, 32, 14, 14)
        x = self.pool(F.relu(self.conv2(x)))   # → (N, 64, 7, 7)
        x = x.view(x.size(0), -1)              # Flatten
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

cnn = SimpleCNN()
print(cnn)
print(f"\nParameter: {sum(p.numel() for p in cnn.parameters()):,}")

## 5. Training & Evaluation

Die Kern-Funktionen für den Trainingsloop:

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Eine Trainings-Epoche."""
    model.train()
    total_loss, correct, total = 0, 0, 0

    for data, target in loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    """Evaluation auf Testdaten."""
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    return total_loss / len(loader), correct / total

## 6. Modelle trainieren

Wir trainieren beide Architekturen und vergleichen sie:

In [ ]:
models = {
    "MLP (784→128→64→10)": SimpleMLP(),
    "CNN (2×Conv+2×FC)": SimpleCNN(),
}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    model = model.to(device)
    print(f"   Parameter: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    epochs = 5

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        print(f"   Epoche {epoch+1}: "
              f"Train Loss={train_loss:.4f} Acc={train_acc:.3f} | "
              f"Test Loss={test_loss:.4f} Acc={test_acc:.3f}")

print("\n✅ Training abgeschlossen!")

## Zusammenfassung

| Konzept | Beschreibung |
|---------|-------------|
| **`nn.Module`** | Basisklasse für alle neuronalen Netze |
| **`nn.Linear`** | Vollvernetzter Layer: y = xW^T + b |
| **`nn.Conv2d`** | 2D-Faltung für räumliche Features |
| **`nn.Dropout`** | Regularisierung gegen Overfitting |
| **`DataLoader`** | Effizientes Batching & Shuffling |
| **`CrossEntropyLoss`** | Kombiniert LogSoftmax + NLLLoss |
| **`optim.Adam`** | Adaptiver Optimierer (momentum + adaptive LR) |
| **`model.train()` / `model.eval()`** | Schaltet Dropout/BatchNorm um |

**Ergebnis:** CNN schlägt MLP auf Bilddaten deutlich — Faltungen lernen räumliche Muster!